# EY Open Science AI & Data Challenge 2026
## Water Quality Prediction — Model Development Description Notebook


---

## 1. Model Development Journey

This section documents every major modeling strategy explored during the challenge.
The progression was driven by a central difficulty visible in the map above:
**submission stations are geographically clustered along the southern coast of South Africa,
while training stations are distributed across the entire country.**
This spatial mismatch made standard cross-validation misleading and forced us to
design spatially-aware training strategies.

---

### 1.1 Phase 1 — Standard Cross-Validation (Baseline)

**Approach:** We first trained models using a standard K-Fold cross-validation on the entire training dataset, with simple models like Random Forest, Gradient Boosting and LightGBoost. the Random forest was the best one with a score of 0.38 (with hyperparameter tuning using Grid search CV)

**Findings:**
- Cross-validation R² scores appeared promising, but leaderboard scores were significantly lower
- Investigation revealed **spatial autocorrelation**: nearby stations share similar water chemistry,
  so a fold containing a station close to a held-out station inflates apparent performance
- Spatial features (coordinates, distance-to-sea, coastal zone) caused **overfitting** —
  the model memorized the geography of training stations rather than learning generalizable patterns

**Conclusion:** Standard CV over-estimated true generalization ability.
Spatial structure had to be explicitly accounted for.

| Aspect | Observation |
|--------|-------------|
| CV R² | Appeared high |
| Leaderboard R² | Much lower — overfitting confirmed |
| Root cause | Spatial autocorrelation between CV folds |
| Decision | Drop naive CV; design spatially-aware strategies |

---

### 1.2 Phase 2 — Nearest-Neighbor Geographic Selection

**Motivation:** Since submission stations are concentrated along the southern coast,
we hypothesized that a model trained only on geographically similar training stations
would generalize better to the submission area.

**Approach:**
1. Computed the **geodesic distance** (Haversine) from every training station to the nearest submission station
2. Ranked training stations by proximity to the submission zone
3. Selected only the closest *k* training stations (by location) to build a geographically restricted training set
4. Trained models on this restricted dataset and evaluated on the leaderboard

```python
from sklearn.neighbors import BallTree
import numpy as np

# Compute distances from training stations to nearest submission station
train_rad = np.radians(station_coords[['Latitude','Longitude']].values)
sub_rad   = np.radians(val_data[['Latitude','Longitude']].values)

tree = BallTree(sub_rad, metric='haversine')
distances, _ = tree.query(train_rad, k=1)
distances_km = distances[:, 0] * 6371  # Earth radius in km

station_coords['dist_to_submission_km'] = distances_km
```

**Result:** Performance did **not improve** on the leaderboard.

**Analysis:** The southern coast stations, while geographically close to submissions,
did not have enough samples to train robust models. Restricting the training set
**reduced data volume** without sufficient gain in distributional alignment.
Water quality drivers (soil, climate, land use) have regional patterns that extend
well beyond geographic proximity alone.

---

### 1.3 Phase 3 — Distance-Weighted Training with Dropout Regularization

**Motivation:** Rather than hard-filtering training data by geography, we designed a softer strategy:
**weight each training sample** by its station's proximity to the submission area,
so that nearby stations contribute more to the model while distant ones are not discarded entirely.

To prevent overfitting from this weighting scheme, we added a **dropout-style cross-validation**:
at each iteration, a random set of *k* location IDs (entire stations) was dropped from training,
and predictions were made on the held-out stations. Final predictions were averaged across iterations.

**Step 1 — Compute sample weights**

```python
# Weight = inverse distance to nearest submission station (closer = higher weight)
# Soft weighting: all stations contribute, but proximal ones are privileged
max_dist = station_coords['dist_to_submission_km'].max()
station_coords['weight'] = 1 - (station_coords['dist_to_submission_km'] / max_dist)
station_coords['weight'] = station_coords['weight'].clip(lower=0.05)  # minimum weight floor

# Merge weights back onto training rows
train_data = train_data.merge(
    station_coords[['location_id','weight']], on='location_id', how='left'
)
```

**Step 2 — Dropout cross-validation loop**

The loop was implemented as a **parameterized function** so that both the number of iterations
and the dropout size could be tuned independently per experiment. In practice, iterations
were set to **15–20 rounds**, and dropout size was adjusted based on the number of available stations.

At each iteration:
- Randomly drop *k* location IDs (entire stations, not individual rows) from the training set
- Train a **per-target model** on the remaining stations, using the distance-based sample weights
- Predict on the submission set and accumulate predictions

Final submission = **average across all iterations** (reduces variance introduced by the dropout).
- Here is axample of implementation using random forest as model for all of the three targets(we didn't record the exact best score of this modeling strategybut it was around 0.44):

```python
def run_dropout_cv(
    train_df,
    submission_df,
    feature_sets,
    target_columns,
    n_iter=20,          # ~15-20 in practice; configurable
    k_dropout=10,       # number of station IDs dropped per round; configurable
    random_seed=42
):
    """
    Distance-weighted training with station-level dropout regularization.
    Each iteration drops k full station IDs, trains per-target models with
    distance-based sample weights, and accumulates predictions.
    Final output is the mean prediction across all iterations.
    """
    rng = np.random.RandomState(random_seed)
    all_station_ids = train_df['location_id'].unique()
    all_predictions = {target: [] for target in target_columns}

    for iteration in range(n_iter):
        # ── Station-level dropout ─────────────────────────────────────────
        drop_ids   = rng.choice(all_station_ids, size=k_dropout, replace=False)
        train_iter = train_df[~train_df['location_id'].isin(drop_ids)].copy()
        weights    = train_iter['weight'].values

        # ── Per-target model (each target has its own model & feature set) ─
        for target in target_columns:
            cols  = feature_sets[target]
            X_tr  = train_iter[cols]
            y_tr  = train_iter[target]
            X_sub = submission_df[cols]

            model = LGBMRegressor(
                n_estimators=400,
                learning_rate=0.05,
                num_leaves=63,
                random_state=iteration  # vary seed per iteration for diversity
            )
            model.fit(X_tr, y_tr, sample_weight=weights)
            all_predictions[target].append(model.predict(X_sub))

    # ── Average predictions across all iterations ─────────────────────────
    return {
        target: np.mean(all_predictions[target], axis=0)
        for target in target_columns
    }


# Example call (parameters were tuned experimentally)
predictions = run_dropout_cv(
    train_df       = train_feat_full,
    submission_df  = submission_feat,
    feature_sets   = feature_sets,
    target_columns = TARGET_COLUMNS,
    n_iter         = 20,
    k_dropout      = 10
)
```

**Result:** This approach showed improvement in some targets, but overall leaderboard gain was modest.
The dropout regularization helped reduce overfitting, but the distance weighting alone was insufficient
to bridge the distributional gap between training and submission stations.

**Key insight:** The submission stations (coastal, eastern Cape) have a distinct
hydroclimatic signature — lower TWI, different soil composition — that requires the model
to interpolate beyond training data, which is fundamentally hard regardless of weighting strategy.

---

### 1.4 Phase 4 — Per-Target Ensemble (Final Strategy)

**Motivation:** Lessons from Phases 1–3 led to a pragmatic conclusion:
use the **full training dataset** (maximum data), apply careful feature engineering
to reduce spatial overfitting, and build **separate ensemble models per target**
with specific feature sets chosen by importance and domain knowledge.

Key decisions:
- Remove spatial coordinate features that caused overfitting in Phase 1
- Replace them with physically meaningful derived features (TWI, soil, climate regimes)
- Train per-target models since TA, EC and DRP have different environmental drivers
- Use weighted ensembles of complementary algorithms to reduce variance

**Leaderboard Progression Summary**

| Phase | Strategy | Key Change | Score |
|-------|----------|-----------|-------|
| 1 | Standard K-Fold CV | Baseline, full features | ~0.35 |
| 1b | Per-target RF | Separate models per target | ~0.42 |
| 2 | Geographic selection | Nearest stations only | No gain |
| 3 | Distance-weighted + Dropout CV | Sample weighting + regularization | ~0.45 |
| 4 | Feature engineering | Rolling windows, interactions, regimes | ~0.46 |
| 4b | Outlier removal |  | ~0.47 |
| **4c** | **Weighted ensemble** | **ExtraTrees + RF + XGB + HGBM per target** | **0.4829** |

---


## 2. Final Model Architecture

### 2.1 Per-Target Feature Sets

Each target has its own curated feature set, selected through two complementary approaches:

1. **Recursive feature addition**: features were added one by one, keeping those that improved
   validation R² for the specific target; others were discarded
2. **Feature importance filtering**: after initial training, features with near-zero importance
   were removed, and the model was retrained on the reduced set

This process revealed that the drivers of TA, EC and DRP are genuinely different:
- **TA** is dominated by terrain (TWI), soil chemistry (pH, CEC), and precipitation regime
- **EC** is driven by climate aridity (dryness ratio), geographic position, and rolling precipitation
- **DRP** is most sensitive to land use (agricultural runoff), soil organic carbon, and runoff index


In [ ]:
# NOTE — REDACTED FOR THIS PUBLIC REPOSITORY.
# The exact per-target feature sets are withheld, as this pipeline
# underlies ongoing (unpublished) research. See docs/Model_Description.md
# for the general feature engineering categories used.
feature_sets = {target: feature_columns for target in target_columns}


### 8.2 Model Training

Each target uses a different ensemble strategy based on what worked best during development:

**Total Alkalinity**: Two ExtraTrees models + HistGradientBoosting  
**Electrical Conductance**: RandomForest + XGBoost + HistGradientBoosting  
**Dissolved Reactive Phosphorus**: RandomForest (standalone — ensembling did not help for this target)


In [ ]:
# NOTE — REDACTED FOR THIS PUBLIC REPOSITORY.
# Exact hyperparameters and ensemble blend weights per target are withheld,
# as this pipeline underlies ongoing (unpublished) research.
# See docs/Model_Description.md for the general modeling approach.
models = {}
predictions = {}
for target in target_columns:
    cols = feature_sets.get(target, feature_columns)
    model = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)
    model.fit(train_feat[cols], train_feat[target])
    preds = np.clip(model.predict(submission_feat[cols]), 0, None)
    models[target] = model
    predictions[target] = preds
